In [2]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("train.csv")
weather = pd.read_csv("weather_data_nyc_centralpark_2016(1).csv")

# =========================
# CLEAN WEATHER DATA
# =========================
weather['date'] = pd.to_datetime(weather['date'], dayfirst=True, errors='coerce')
weather = weather.dropna(subset=['date'])
weather['date'] = weather['date'].dt.date

# Replace 'T' (trace) with small value
weather = weather.replace('T', 0.01)

# Convert numeric columns
cols = ['average temperature', 'precipitation', 'snow fall']
for col in cols:
    weather[col] = pd.to_numeric(weather[col], errors='coerce')

# Rename columns
weather = weather.rename(columns={
    'average temperature': 'avg_temp',
    'precipitation': 'precip',
    'snow fall': 'snow'
})

# Keep only needed columns
weather = weather[['date', 'avg_temp', 'precip', 'snow']]

# =========================
# PREPARE TAXI DATA
# =========================
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['date'] = df['pickup_datetime'].dt.date
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek

# =========================
# MERGE WEATHER
# =========================
df = df.merge(weather, on='date', how='left')

# =========================
# BASIC CLEANING
# =========================
df = df[df['trip_duration'] > 0]
df = df[df['passenger_count'] > 0]

# Remove coordinate outliers (NYC bounds)
df = df[
    (df['pickup_latitude'].between(40.5, 41)) &
    (df['pickup_longitude'].between(-74.5, -73)) &
    (df['dropoff_latitude'].between(40.5, 41)) &
    (df['dropoff_longitude'].between(-74.5, -73))
]

# =========================
# DISTANCE (HAVERSINE)
# =========================
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371
    return c * r

df['distance'] = df.apply(lambda row: haversine(
    row['pickup_longitude'], row['pickup_latitude'],
    row['dropoff_longitude'], row['dropoff_latitude']
), axis=1)

# =========================
# SPEED (km/h)
# =========================
df['speed'] = df['distance'] / (df['trip_duration'] / 3600)

# Remove unrealistic speeds
df = df[(df['speed'] > 0) & (df['speed'] < 120)]

# =========================
# FINAL CLEANING
# =========================
df = df.rename(columns={
    'pickup_latitude': 'pickup_lat',
    'pickup_longitude': 'pickup_lon',
    'dropoff_latitude': 'drop_lat',
    'dropoff_longitude': 'drop_lon'
})

# Drop rows where weather still missing (rare)
df = df.dropna(subset=['avg_temp', 'precip', 'snow'])

# =========================
# FINAL MASTER DATASET
# =========================
df = df[[
    'pickup_lat','pickup_lon',
    'drop_lat','drop_lon',
    'hour','day_of_week',
    'passenger_count',
    'avg_temp','precip','snow',
    'speed'
]]

# =========================
# SAVE CLEAN DATASET
# =========================

print("Final dataset shape:", df.shape)
print(df.head())

Final dataset shape: (1452287, 11)
   pickup_lat  pickup_lon   drop_lat   drop_lon  hour  day_of_week  \
0   40.767937  -73.982155  40.765602 -73.964630    17            0   
1   40.738564  -73.980415  40.731152 -73.999481     0            6   
2   40.763939  -73.979027  40.710087 -74.005333    11            1   
3   40.719971  -74.010040  40.706718 -74.012268    19            2   
4   40.793209  -73.973053  40.782520 -73.972923    13            5   

   passenger_count  avg_temp  precip  snow      speed  
0                1      45.5    0.29   0.0  11.856428  
1                1      72.5    0.00   0.0   9.803659  
2                1      22.0    0.00   0.0  10.822201  
3                1      39.0    0.00   0.0  12.465721  
4                1      46.5    0.00   0.0   9.836594  


In [3]:
df = df.rename(columns={'snow': 'snowfall'})

In [6]:
df['is_snow'] = (df['snowfall'] > 0).astype(int)

In [7]:
df = df.drop(columns=['snowfall'])

In [8]:
df['is_snow'].value_counts()

is_snow
0    1301361
1     150926
Name: count, dtype: int64

In [9]:
df.to_csv("final_preprocessed_dataset.csv", index=False)
